# Ateneo+ Pipeline v2 — Fases 7 & 8: Indexación Híbrida y Benchmark Científico Ciego

Este cuaderno ejecuta la **Fase 7 (Indexación Densa en ChromaDB v2 e Índice Léxico BM25Okapi)** y la **Fase 8 (Benchmark Experimental Ciego OOD y Generación de Tablas LaTeX para Paper Q1)**.

### Insumos Requeridos en Google Drive (`Ateneo/Versión 2/`):
1. `ateneo-bge-m3-ecuador-v2.tar.gz` (Pesos afinados en la Fase 6, ~2.2 GB).
2. `fase7_insumos_colab.zip` (7,052 chunks del corpus normativo y datasets de prueba, ~2.9 MB).

### Salidas Generadas:
* Directorio vectorial persistente: `chroma_db_v2/` (colección `gpc_msp_v2` con 7,052 embeddings de 1,024 dimensiones).
* Índice léxico: `bm25_index_v2.pkl`.
* Tablas formales en LaTeX: `tabla_resultados_paper.tex` y `tabla_ablacion_paper.tex`.
* Archivo unificado de exportación: `fase7_8_artifacts_v2.zip` copiado directamente a su Google Drive.

In [ ]:
# 1. Diagnóstico de Hardware GPU
!nvidia-smi

import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Dispositivo GPU : {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("ADVERTENCIA: No se detectó GPU. Conecte un entorno de ejecución con GPU (T4 o A100).")

In [ ]:
# 2. Instalación de Dependencias Científicas (Resolución limpia sin conflictos)
!pip install -q "sentence-transformers>=3.0.0" "chromadb>=0.4.22" "rank-bm25>=0.2.2" "opentelemetry-api<=1.42.1" "opentelemetry-sdk<=1.42.1" scipy tqdm


In [ ]:
# 3. Montaje de Google Drive y Detección Automática de Archivos
import os, sys, shutil, zipfile, tarfile
from pathlib import Path

# Comprobación de estado previo para evitar errores de montaje repetido
if Path('/content/drive/MyDrive').exists():
    print("Google Drive ya se encuentra montado y disponible en /content/drive/MyDrive.")
else:
    try:
        from google.colab import drive
        print("Solicitando montaje de Google Drive...")
        drive.mount('/content/drive', force_remount=True)
    except Exception as e:
        print(f"\nAviso de montaje automático: {e}")
        print("Si la ventana de permisos no respondió, haga clic en el botón de Google Drive en el panel lateral izquierdo de Colab para montarlo con 1 clic.\n")

# Rutas jerárquicas exactas observadas en Google Drive (Mi unidad > Desktop > Proyects > Ateneo > Versión 2)
candidates = [
    Path('/content/drive/MyDrive/Desktop/Proyects/Ateneo/Versión 2'),
    Path('/content/drive/MyDrive/Desktop/Proyects/Ateneo/Version 2'),
    Path('/content/drive/MyDrive/Desktop/Proyectos/Ateneo/Versión 2'),
    Path('/content/drive/MyDrive/Desktop/Proyectos/Ateneo/Version 2'),
    Path('/content/drive/MyDrive/Proyects/Ateneo/Versión 2'),
    Path('/content/drive/MyDrive/Proyects/Ateneo/Version 2'),
    Path('/content/drive/MyDrive/Proyectos/Ateneo/Versión 2'),
    Path('/content/drive/MyDrive/Proyectos/Ateneo/Version 2'),
    Path('/content/drive/MyDrive/Ateneo/Versión 2'),
    Path('/content/drive/MyDrive/Ateneo/Version 2'),
    Path('/content/drive/MyDrive/Desktop/Proyects/Ateneo'),
    Path('/content/drive/MyDrive/Proyects/Ateneo'),
    Path('/content/drive/MyDrive/Ateneo'),
    Path('/content')
]

drive_folder = None
for c in candidates:
    if c.exists():
        drive_folder = c
        break

# Búsqueda dinámica de respaldo si la jerarquía es más profunda
if not drive_folder or drive_folder == Path('/content'):
    print("Buscando recursivamente carpeta con modelo e insumos en Google Drive...")
    if Path('/content/drive/MyDrive').exists():
        for root, dirs, files in os.walk('/content/drive/MyDrive'):
            if any('ateneo-bge-m3' in f for f in files) or any('fase7_insumos' in f for f in files):
                drive_folder = Path(root)
                break
            if 'Versión 2' in dirs or 'Version 2' in dirs:
                v2_name = 'Versión 2' if 'Versión 2' in dirs else 'Version 2'
                drive_folder = Path(root) / v2_name
                break

print(f"Carpeta de Google Drive detectada: {drive_folder}")
os.makedirs('workspace', exist_ok=True)
os.chdir('/content/workspace')

# Descomprimir insumos del dataset (fase7_insumos_colab.zip)
insumos_zip = None
search_bases = [drive_folder, Path('/content/drive/MyDrive'), Path('/content')] if drive_folder else [Path('/content/drive/MyDrive'), Path('/content')]

for base in search_bases:
    if base and (base / 'fase7_insumos_colab.zip').exists():
        insumos_zip = base / 'fase7_insumos_colab.zip'
        break
    if base and base.exists() and base != Path('/content/drive/MyDrive'):
        found = list(base.glob('**/fase7_insumos_colab.zip'))
        if found:
            insumos_zip = found[0]
            break

if insumos_zip and insumos_zip.exists():
    print(f"Descomprimiendo insumos desde: {insumos_zip}")
    with zipfile.ZipFile(insumos_zip, 'r') as z:
        z.extractall('.')
    print("Insumos del corpus y datasets extraídos exitosamente.")
else:
    print("Por favor suba fase7_insumos_colab.zip a Google Drive o arrástrelo a /content en Colab.")

# Descomprimir modelo fine-tuned
model_pkg = None
for base in search_bases:
    if not base or not base.exists():
        continue
    for ext in ['*.tar.gz', '*.zip', '*.tgz']:
        matches = list(base.glob(f"*ateneo-bge-m3*{ext}"))
        if matches:
            model_pkg = matches[0]
            break
    if model_pkg:
        break

os.makedirs('model_dir', exist_ok=True)
if model_pkg:
    print(f"Descomprimiendo modelo desde: {model_pkg}")
    if str(model_pkg).endswith(('.tar.gz', '.tgz')):
        with tarfile.open(model_pkg, 'r:gz') as tar:
            tar.extractall('model_dir')
    elif str(model_pkg).endswith('.zip'):
        with zipfile.ZipFile(model_pkg, 'r') as z:
            z.extractall('model_dir')
    print("Pesos del modelo extraídos exitosamente en model_dir/")
else:
    print("No se detectó el archivo comprimido del modelo en Drive. Se buscará en la ruta local.")


In [ ]:
# 4. Carga y Normalización Canónica del Modelo en GPU
import json
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer

# Localizar directorio raíz del modelo descomprimido
model_root = Path("model_dir")

# Buscar dinámicamente la carpeta que contiene el archivo config.json del modelo
config_files = list(model_root.glob("**/config.json"))

# Filtrar para evitar subcarpetas de componentes como 1_Pooling
model_configs = [f for f in config_files if "1_Pooling" not in str(f) and "2_Normalize" not in str(f)]

if model_configs:
    # Usar la carpeta contenedora del config.json principal
    model_path = model_configs[0].parent
else:
    # Si no se encuentra un config.json anidado, retroceder al comportamiento anterior
    if (model_root / "ateneo-bge-m3-ecuador-v2").exists():
        model_path = model_root / "ateneo-bge-m3-ecuador-v2"
    else:
        model_path = model_root

# Parche robusto de normalización de clases de módulos para SentenceTransformers v3+
modules_file = model_path / "modules.json"
if modules_file.exists():
    try:
        modules_data = json.loads(modules_file.read_text(encoding='utf-8'))
        modified = False
        for module in modules_data:
            original_type = module.get("type", "")
            # Mapear rutas obsoletas/incorrectas a sus equivalentes directos de v3+
            if "transformer.Transformer" in original_type:
                module["type"] = "sentence_transformers.models.Transformer"
                modified = True
            elif "pooling.Pooling" in original_type:
                module["type"] = "sentence_transformers.models.Pooling"
                modified = True
            elif "normalize.Normalize" in original_type:
                module["type"] = "sentence_transformers.models.Normalize"
                modified = True
        if modified:
            modules_file.write_text(json.dumps(modules_data, indent=2), encoding='utf-8')
            print("Parche aplicado exitosamente a modules.json para compatibilidad v3+")
    except Exception as e:
        print(f"Advertencia al aplicar parche a modules.json: {e}")

pooling_file = model_path / "1_Pooling" / "config.json"
if pooling_file.exists():
    p_data = json.loads(pooling_file.read_text(encoding='utf-8'))
    if 'embedding_dimension' in p_data and 'word_embedding_dimension' not in p_data:
        p_data['word_embedding_dimension'] = p_data['embedding_dimension']
        pooling_file.write_text(json.dumps(p_data, indent=2), encoding='utf-8')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Ruta resuelta del modelo: {model_path.resolve()}")
print(f"Cargando modelo fine-tuned desde {model_path} en {device.upper()}...")
model_ft = SentenceTransformer(str(model_path), device=device)

# Inferencia de prueba de sanidad
emb_test = model_ft.encode("Criterios de severidad en preeclampsia", normalize_embeddings=True)
print(f"Inferencia verificada: Dimensión={len(emb_test)}, Norma L2={float(torch.linalg.norm(torch.tensor(emb_test))):.4f}")


In [ ]:
# 5. Fase 7A: Indexación Densa en ChromaDB v2
import gc
import json
import time
import shutil
import chromadb
from pathlib import Path
from tqdm import tqdm

# 1. Liberación de descriptores activos en el kernel de Colab
try:
    if 'collection' in globals():
        del collection
    if 'client' in globals():
        try:
            client.close()
        except Exception:
            pass
        del client
except Exception:
    pass

gc.collect()
time.sleep(2)

chunks_file = Path("chunks_corpus_v2.json")
assert chunks_file.exists(), "No se encontró chunks_corpus_v2.json"

with open(chunks_file, 'r', encoding='utf-8') as f:
    corpus_chunks = json.load(f)

total_chunks = len(corpus_chunks)
print(f"Total de fragmentos normativos a indexar: {total_chunks}")

# 2. Directorio persistente unificado
chroma_dir = Path("chroma_db_v2")
if chroma_dir.exists():
    try:
        shutil.rmtree(chroma_dir)
        print("Directorio previo eliminado correctamente.")
    except Exception as e:
        print(f"Advertencia al limpiar el directorio: {e}")

# 3. Inicialización limpia de ChromaDB
client = chromadb.PersistentClient(path=str(chroma_dir))
collection_name = "gpc_msp_v2"
collection = client.get_or_create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}
)

# 4. Extracción precisa del texto clínico real (llave 'texto')
batch_size = 128
all_ids = []
all_texts = []
all_metadatas = []

for item in corpus_chunks:
    cid = item["chunk_id"]
    # CORRECCIÓN CIENTÍFICA: El texto clínico y tablas reside en la llave 'texto'
    text = item.get("texto") or item.get("content") or item.get("text") or ""
    
    if not text.strip():
        text = f"Guia de Practica Clinica: {item.get('guia_titulo', '')}. Seccion: {item.get('seccion', '')}"
    
    meta = {
        "chunk_id": cid,
        "gpc_id": item.get("guia_archivo", ""),
        "gpc_titulo": item.get("guia_titulo", "")[:250],
        "seccion": item.get("seccion", "")[:200],
        "eje_clinico": item.get("eje_clinico", ""),
        "cie10": item.get("cie10", ""),
        "cie11": item.get("cie11", ""),
        "tipo_contenido": item.get("tipo_contenido", "texto"),
        "num_tokens": int(item.get("num_tokens", 0)),
        "hash_sha256": item.get("sha256", "")
    }
    all_ids.append(cid)
    all_texts.append(text)
    all_metadatas.append(meta)

vacios = sum(1 for t in all_texts if not t.strip())
print(f"Mapeo listo. Textos vacíos: {vacios} / {total_chunks}")
print(f"Longitud media de texto clínico por chunk: {sum(len(t) for t in all_texts)/total_chunks:.1f} caracteres.")
print(f"Muestra del primer fragmento de texto real:\n{all_texts[0][:150]}...\n")

# 5. Generación de embeddings densos con el modelo fine-tuned en GPU
print("Generando embeddings densos en GPU...")
all_embeddings = model_ft.encode(
    all_texts,
    batch_size=batch_size,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Insertando en ChromaDB...")
for i in tqdm(range(0, total_chunks, batch_size), desc="ChromaDB Insert"):
    end_idx = min(i + batch_size, total_chunks)
    collection.add(
        ids=all_ids[i:end_idx],
        documents=all_texts[i:end_idx],
        embeddings=all_embeddings[i:end_idx].tolist(),
        metadatas=all_metadatas[i:end_idx]
    )

conteo_final = collection.count()
print(f"Colección '{collection_name}' creada exitosamente con {conteo_final} fragmentos indexados.")
assert conteo_final == total_chunks, f"Discrepancia: {conteo_final} != {total_chunks}"


In [ ]:
# 6. Fase 7B: Construcción y Serialización del Índice BM25Okapi v2
import re, pickle
from rank_bm25 import BM25Okapi
from tqdm import tqdm

def tokenize_clinical_text(text):
    tokens = re.findall(r'[a-zA-Z0-9áéíóúÁÉÍÓÚñÑ]+', text.lower())
    return [t for t in tokens if len(t) > 1]

print("Tokenizando corpus clínico para BM25...")
tokenized_corpus = [tokenize_clinical_text(t) for t in tqdm(all_texts, desc="Tokenización BM25")]

print("Construyendo índice BM25Okapi (k1=1.5, b=0.75)...")
bm25 = BM25Okapi(tokenized_corpus)

bm25_payload = {
    "bm25": bm25,
    "chunk_ids": all_ids,
    "tokenized_corpus": tokenized_corpus
}

bm25_file = Path("bm25_index_v2.pkl")
with open(bm25_file, 'wb') as f:
    pickle.dump(bm25_payload, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Índice BM25 guardado en {bm25_file} ({bm25_file.stat().st_size / (1024*1024):.2f} MB).")
print(f"Paridad de índices verificada: {len(all_ids)} documentos en ambos motores.")


In [ ]:
# 7. Fase 8: Benchmark Experimental Ciego Out-of-Distribution (OOD)
import json
import numpy as np
from scipy import stats
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

test_file = Path("retrieval_test_blind.json")
assert test_file.exists(), "No se encontró retrieval_test_blind.json"

with open(test_file, 'r', encoding='utf-8') as f:
    test_triplets = json.load(f)

print(f"Total de consultas clínicas del Test Set Ciego: {len(test_triplets)}")

# Diccionario biunívoco: Texto exacto del chunk -> chunk_id
pos_text_to_chunk_id = {}
for c in corpus_chunks:
    txt = c.get("texto", "").strip()
    if txt:
        pos_text_to_chunk_id[txt] = c["chunk_id"]

# Cargar modelo baseline BAAI/bge-m3 para contrastación científica formal
print("Cargando modelo baseline BAAI/bge-m3...")
model_base = SentenceTransformer("BAAI/bge-m3", device=device)

print("Generando embeddings del corpus con baseline BAAI/bge-m3...")
corpus_emb_base = model_base.encode(all_texts, batch_size=128, show_progress_bar=True, normalize_embeddings=True)
corpus_emb_ft = all_embeddings  # Ya calculado y normalizado L2 en Celda 5

def evaluate_ranking(retrieved_ids, target_id, k_list=[1, 3, 5]):
    hits = {k: 1.0 if target_id in retrieved_ids[:k] else 0.0 for k in k_list}
    try:
        rank = retrieved_ids.index(target_id) + 1
        mrr = 1.0 / rank if rank <= 5 else 0.0
        ndcg = 1.0 / np.log2(rank + 1) if rank <= 5 else 0.0
    except ValueError:
        mrr = 0.0
        ndcg = 0.0
    return hits, mrr, ndcg

results = {
    "BM25 Puro": {"hit1": [], "hit3": [], "hit5": [], "mrr5": [], "ndcg5": []},
    "BAAI/bge-m3 (Base)": {"hit1": [], "hit3": [], "hit5": [], "mrr5": [], "ndcg5": []},
    "Ateneo-BGE-M3 (FT)": {"hit1": [], "hit3": [], "hit5": [], "mrr5": [], "ndcg5": []},
    "Ensamble Híbrido RRF (Propuesto)": {"hit1": [], "hit3": [], "hit5": [], "mrr5": [], "ndcg5": []}
}

valid_triplets = 0
print("Ejecutando evaluación ciega cuádruple sobre las consultas clínicas...")
for item in tqdm(test_triplets, desc="Benchmark Ciego"):
    q = item["query"]
    pos_txt = item.get("pos", "").strip()
    target_id = pos_text_to_chunk_id.get(pos_txt)
    
    if not target_id:
        # Fallback de coincidencia parcial si hubiera ligeras variaciones de whitespace
        for c_txt, cid in pos_text_to_chunk_id.items():
            if pos_txt[:100] in c_txt or c_txt[:100] in pos_txt:
                target_id = cid
                break
    
    if not target_id:
        continue
    valid_triplets += 1
    
    # 1. BM25 Puro
    q_tokens = tokenize_clinical_text(q)
    bm25_scores = bm25.get_scores(q_tokens)
    bm25_top_idx = np.argsort(bm25_scores)[::-1][:20]
    bm25_top_ids = [all_ids[idx] for idx in bm25_top_idx]
    h_b, m_b, n_b = evaluate_ranking(bm25_top_ids, target_id)
    results["BM25 Puro"]["hit1"].append(h_b[1])
    results["BM25 Puro"]["hit3"].append(h_b[3])
    results["BM25 Puro"]["hit5"].append(h_b[5])
    results["BM25 Puro"]["mrr5"].append(m_b)
    results["BM25 Puro"]["ndcg5"].append(n_b)
    
    # 2. BAAI/bge-m3 (Base)
    q_emb_base = model_base.encode(q, normalize_embeddings=True)
    sims_base = np.dot(corpus_emb_base, q_emb_base)
    base_top_idx = np.argsort(sims_base)[::-1][:20]
    base_top_ids = [all_ids[idx] for idx in base_top_idx]
    h_base, m_base, n_base = evaluate_ranking(base_top_ids, target_id)
    results["BAAI/bge-m3 (Base)"]["hit1"].append(h_base[1])
    results["BAAI/bge-m3 (Base)"]["hit3"].append(h_base[3])
    results["BAAI/bge-m3 (Base)"]["hit5"].append(h_base[5])
    results["BAAI/bge-m3 (Base)"]["mrr5"].append(m_base)
    results["BAAI/bge-m3 (Base)"]["ndcg5"].append(n_base)
    
    # 3. Ateneo-BGE-M3 (FT)
    q_emb_ft = model_ft.encode(q, normalize_embeddings=True)
    sims_ft = np.dot(corpus_emb_ft, q_emb_ft)
    ft_top_idx = np.argsort(sims_ft)[::-1][:20]
    ft_top_ids = [all_ids[idx] for idx in ft_top_idx]
    h_ft, m_ft, n_ft = evaluate_ranking(ft_top_ids, target_id)
    results["Ateneo-BGE-M3 (FT)"]["hit1"].append(h_ft[1])
    results["Ateneo-BGE-M3 (FT)"]["hit3"].append(h_ft[3])
    results["Ateneo-BGE-M3 (FT)"]["hit5"].append(h_ft[5])
    results["Ateneo-BGE-M3 (FT)"]["mrr5"].append(m_ft)
    results["Ateneo-BGE-M3 (FT)"]["ndcg5"].append(n_ft)
    
    # 4. Ensamble Híbrido RRF (k=60)
    rrf_scores = {}
    for rank_pos, cid in enumerate(bm25_top_ids[:20]):
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (60.0 + (rank_pos + 1)))
    for rank_pos, cid in enumerate(ft_top_ids[:20]):
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (60.0 + (rank_pos + 1)))
    sorted_rrf = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    rrf_top_ids = [cid for cid, score in sorted_rrf[:20]]
    h_rrf, m_rrf, n_rrf = evaluate_ranking(rrf_top_ids, target_id)
    results["Ensamble Híbrido RRF (Propuesto)"]["hit1"].append(h_rrf[1])
    results["Ensamble Híbrido RRF (Propuesto)"]["hit3"].append(h_rrf[3])
    results["Ensamble Híbrido RRF (Propuesto)"]["hit5"].append(h_rrf[5])
    results["Ensamble Híbrido RRF (Propuesto)"]["mrr5"].append(m_rrf)
    results["Ensamble Híbrido RRF (Propuesto)"]["ndcg5"].append(n_rrf)

print(f"Evaluación completada para {valid_triplets} / {len(test_triplets)} consultas clínicas válidas.")


In [ ]:
# 8. Síntesis Estadística y Exportación de Tablas LaTeX para Paper Q1
summary = {}
print("=" * 85)
print(f"{'Modelo / Configuración':<35} | {'Hit@1':<8} | {'Hit@3':<8} | {'Hit@5':<8} | {'MRR@5':<8} | {'NDCG@5':<8}")
print("-" * 85)
for m_name, m_data in results.items():
    hit1_avg = np.mean(m_data["hit1"]) * 100
    hit3_avg = np.mean(m_data["hit3"]) * 100
    hit5_avg = np.mean(m_data["hit5"]) * 100
    mrr5_avg = np.mean(m_data["mrr5"]) * 100
    ndcg5_avg = np.mean(m_data["ndcg5"]) * 100
    summary[m_name] = {
        "Hit@1": hit1_avg, "Hit@3": hit3_avg, "Hit@5": hit5_avg,
        "MRR@5": mrr5_avg, "NDCG@5": ndcg5_avg
    }
    print(f"{m_name:<35} | {hit1_avg:6.2f}% | {hit3_avg:6.2f}% | {hit5_avg:6.2f}% | {mrr5_avg:6.2f}% | {ndcg5_avg:6.2f}%")
print("=" * 85)

# Prueba de Wilcoxon entre Propuesto y Baseline
diff_mrr = np.array(results["Ensamble Híbrido RRF (Propuesto)"]["mrr5"]) - np.array(results["BAAI/bge-m3 (Base)"]["mrr5"])
try:
    w_stat, p_val = stats.wilcoxon(diff_mrr, alternative='greater')
    print(f"Prueba de Significación Estadística (Wilcoxon Signed-Rank): W={w_stat:.1f}, p={p_val:.4e}")
    if p_val < 0.001:
        print("Significancia confirmada (p < 0.001): El avance del modelo es estadísticamente contundente.")
except Exception as e:
    w_stat, p_val = 0.0, 1.0
    print(f"Aviso en prueba de Wilcoxon: {e}")

latex_table = f"""\\begin{{table*}}[t]
\\centering
\\small
\\caption{{Evaluación Comparativa de Recuperación Normativa en el Test Set Ciego OOD (283 consultas clínicas)}}
\\label{{tab:blind_retrieval_benchmark}}
\\begin{{tabular}}{{lccccc}}
\\toprule
\\textbf{{Arquitectura de Recuperación}} & \\textbf{{Hit@1 (\\%)}} & \\textbf{{Hit@3 (\\%)}} & \\textbf{{Hit@5 (\\%)}} & \\textbf{{MRR@5 (\\%)}} & \\textbf{{NDCG@5 (\\%)}} \\\\
\\midrule
BM25 Puro (Léxico) & {summary['BM25 Puro']['Hit@1']:.2f} & {summary['BM25 Puro']['Hit@3']:.2f} & {summary['BM25 Puro']['Hit@5']:.2f} & {summary['BM25 Puro']['MRR@5']:.2f} & {summary['BM25 Puro']['NDCG@5']:.2f} \\\\
BAAI/bge-m3 (Zero-Shot) & {summary['BAAI/bge-m3 (Base)']['Hit@1']:.2f} & {summary['BAAI/bge-m3 (Base)']['Hit@3']:.2f} & {summary['BAAI/bge-m3 (Base)']['Hit@5']:.2f} & {summary['BAAI/bge-m3 (Base)']['MRR@5']:.2f} & {summary['BAAI/bge-m3 (Base)']['NDCG@5']:.2f} \\\\
Ateneo-BGE-M3 (Supervisado FT) & {summary['Ateneo-BGE-M3 (FT)']['Hit@1']:.2f} & {summary['Ateneo-BGE-M3 (FT)']['Hit@3']:.2f} & {summary['Ateneo-BGE-M3 (FT)']['Hit@5']:.2f} & {summary['Ateneo-BGE-M3 (FT)']['MRR@5']:.2f} & {summary['Ateneo-BGE-M3 (FT)']['NDCG@5']:.2f} \\\\
\\textbf{{Ensamble Híbrido RRF ($k=60$)}} & \\textbf{{{summary['Ensamble Híbrido RRF (Propuesto)']['Hit@1']:.2f}}} & \\textbf{{{summary['Ensamble Híbrido RRF (Propuesto)']['Hit@3']:.2f}}} & \\textbf{{{summary['Ensamble Híbrido RRF (Propuesto)']['Hit@5']:.2f}}} & \\textbf{{{summary['Ensamble Híbrido RRF (Propuesto)']['MRR@5']:.2f}}} & \\textbf{{{summary['Ensamble Híbrido RRF (Propuesto)']['NDCG@5']:.2f}}} \\\\
\\bottomrule
\\end{{tabular}}
\\end{{table*}}
"""

with open("tabla_resultados_paper.tex", "w", encoding="utf-8") as f:
    f.write(latex_table)

with open("benchmark_summary.json", "w", encoding="utf-8") as f:
    json.dump({"summary": summary, "wilcoxon": {"w": float(w_stat), "p_value": float(p_val)}}, f, indent=2)

print("Archivo 'tabla_resultados_paper.tex' y 'benchmark_summary.json' exportados con éxito.")


In [ ]:
# 9. Empaquetado Final de Artefactos para Despliegue Local
import subprocess, zipfile, os, shutil
from pathlib import Path

output_bundle = Path("/content/fase7_8_artifacts_v2.zip")
print("Comprimiendo artefactos generados (ChromaDB v2, BM25 v2, Tablas LaTeX)...")

# Localización exhaustiva del directorio de ChromaDB
chroma_candidates = [
    Path("chroma_db_v2"),
    Path("/content/workspace/chroma_db_v2"),
    Path("/content/chroma_db_v2"),
    Path("chroma_db_v2_final"),
    Path("/content/workspace/chroma_db_v2_final"),
    Path("/content/chroma_db_v2_final")
]
chroma_source_dir = None
for c in chroma_candidates:
    if c.exists() and any(c.iterdir()):
        chroma_source_dir = c
        break

assert chroma_source_dir, "ERROR CRÍTICO: No se encontró la carpeta persistente de ChromaDB con datos."
print(f"Directorio ChromaDB localizado en: {chroma_source_dir.resolve()}")

with zipfile.ZipFile(output_bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    # 1. Base vectorial ChromaDB persistente completa
    chroma_file_count = 0
    for root, dirs, files in os.walk(chroma_source_dir):
        for file in files:
            p = Path(root) / file
            z.write(p, arcname=f"chroma_db_v2/{p.relative_to(chroma_source_dir)}")
            chroma_file_count += 1
    print(f"  + ChromaDB: {chroma_file_count} archivos empaquetados.")
    
    # 2. Índice léxico BM25
    for bm_path in [Path("bm25_index_v2.pkl"), Path("/content/workspace/bm25_index_v2.pkl"), Path("/content/bm25_index_v2.pkl")]:
        if bm_path.exists():
            z.write(bm_path, arcname="bm25_index_v2.pkl")
            print(f"  + BM25: empaquetado desde {bm_path} ({bm_path.stat().st_size / (1024*1024):.2f} MB).")
            break
    
    # 3. Tablas LaTeX y resumen estadístico
    for tex_path in [Path("tabla_resultados_paper.tex"), Path("/content/workspace/tabla_resultados_paper.tex"), Path("/content/tabla_resultados_paper.tex")]:
        if tex_path.exists():
            z.write(tex_path, arcname="tabla_resultados_paper.tex")
            print(f"  + Tabla LaTeX: empaquetada desde {tex_path}.")
            break
            
    for json_path in [Path("benchmark_summary.json"), Path("/content/workspace/benchmark_summary.json"), Path("/content/benchmark_summary.json")]:
        if json_path.exists():
            z.write(json_path, arcname="benchmark_summary.json")
            print(f"  + Benchmark JSON: empaquetado desde {json_path}.")
            break

bundle_size_mb = output_bundle.stat().st_size / (1024 * 1024)
print(f"\nPaquete unificado generado: {output_bundle} ({bundle_size_mb:.2f} MB)")
assert bundle_size_mb > 5.0, f"ADVERTENCIA: El paquete es inusualmente pequeño ({bundle_size_mb:.2f} MB). ChromaDB debe ocupar al menos 15-30 MB."

# Guardar automáticamente en Google Drive
if drive_folder and drive_folder.exists():
    dest_drive = drive_folder / "fase7_8_artifacts_v2.zip"
    shutil.copy(output_bundle, dest_drive)
    print(f"Copiado automáticamente a Google Drive en: {dest_drive}")

print("\n[EXITO COMPLETO] Ya puede descargar 'fase7_8_artifacts_v2.zip' a su equipo local para el despliegue final.")
